In [1]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

import pandas as pd

import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

In [2]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params = {
	"latitude": 48.1374,
	"longitude": 11.5755,
	"hourly": ["temperature_2m", "relative_humidity_2m", "rain", "surface_pressure"],
	"models": "dwd_icon_seamless",
	"start_date": "2025-01-01",
	"end_date": "2025-12-31",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_rain = hourly.Variables(2).ValuesAsNumpy()
hourly_surface_pressure = hourly.Variables(3).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["rain"] = hourly_rain
hourly_data["surface_pressure"] = hourly_surface_pressure

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Coordinates: 48.13999938964844°N 11.579999923706055°E
Elevation: 524.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
                           date  temperature_2m  relative_humidity_2m  rain  \
0    2025-01-01 00:00:00+00:00         -0.6045                  77.0   0.0   
1    2025-01-01 01:00:00+00:00         -0.8545                  77.0   0.0   
2    2025-01-01 02:00:00+00:00         -2.0545                  83.0   0.0   
3    2025-01-01 03:00:00+00:00         -2.7545                  83.0   0.0   
4    2025-01-01 04:00:00+00:00         -3.7045                  87.0   0.0   
...                        ...             ...                   ...   ...   
8755 2025-12-31 19:00:00+00:00         -2.6545                  63.0   0.0   
8756 2025-12-31 20:00:00+00:00         -3.0545                  63.0   0.0   
8757 2025-12-31 21:00:00+00:00         -3.1545                  66.0   0.0   
8758 2025-12-31 22:00:00+00:00         -3.3545                  64.0   0.0   
8759 2025-12-31 23

In [5]:
hourly_dataframe.columns

Index(['date', 'temperature_2m', 'relative_humidity_2m', 'rain',
       'surface_pressure'],
      dtype='str')

In [3]:
x = hourly_dataframe.index

y = hourly_dataframe["temperature_2m"]

fig = px.scatter(x=x, y=y, labels={'x': 'Hours', 'y': 'Temperature'}, title='')

fig.add_trace(trace=go.Scatter(x=x,y=[np.mean(y).astype(float)]*len(x)))

# Show the plot
fig.show()

In [7]:
hourly_dataframe.to_csv("../data/raw/historical_training_data_2025")

In [4]:
hourly_dataframe.to_parquet(
    "../data/processed/historical_training_data_2025.parquet",
    index=False
)